<a href="https://colab.research.google.com/github/raimundasantos-ship-it/Atividades/blob/main/notebooks/capitulo-01.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Capítulo 1 — Análise Exploratória de Dados

Notebook com o **código** deste capítulo, para o Google Colab. Cada trecho vem precedido de uma explicação curta; o texto completo está no site do livro.

Rode a célula de **setup** abaixo primeiro (uma vez), depois as demais em ordem.

In [1]:
# Setup (rode uma vez).
!pip install -q wquantiles
!curl -sO https://raw.githubusercontent.com/BragaD/UnDF-Bases3-Estatistica-202602/main/formato.py   # baixa o ajudante de formatação do livro

## 1.1 — Elementos de Dados Estruturados

Importa o pandas e configura a exibição para mostrar todas as colunas das tabelas, sem truncar.

In [ ]:
import pandas as pd

pd.set_option("display.max_columns", None)

Carrega os dados dos estados e mostra os tipos que o pandas inferiu sozinho: `Populacao` como número inteiro e `Taxa.Homicidios` como decimal, mas `Estado` e `Sigla` como texto genérico — o pandas não sabe que essas colunas são categóricas.

In [ ]:
estado = pd.read_csv("https://raw.githubusercontent.com/BragaD/UnDF-Bases3-Estatistica-202602/main/dados/estados.csv")
estado.dtypes

Declara `Sigla` como categórica (`category`), fechando o conjunto em 27 valores possíveis. O ganho aqui é semântico — dizer que a coluna é categórica, não texto livre —, não de memória: essa economia só aparece quando poucos valores se repetem em muitas linhas.

In [ ]:
estado["Sigla"] = estado["Sigla"].astype("category")
print(estado["Sigla"].dtype)
print("categorias:", len(estado["Sigla"].cat.categories))

Cria uma coluna categórica **ordenada** (pequeno < médio < grande) e mostra três operações que só funcionam por causa disso: ordenar pela ordem declarada (não pelo alfabeto), comparar com `>` e tirar o máximo.

In [ ]:
from pandas.api.types import CategoricalDtype

tamanho = CategoricalDtype(categories=["pequeno", "medio", "grande"], ordered=True)
s = pd.Series(["grande", "pequeno", "medio"], dtype=tamanho)

print("ordenado:", s.sort_values().tolist())
print("maior que 'pequeno'?", s.gt("pequeno").tolist())
print("máximo:", s.max())

Repete o mesmo dado, mas como categórico **nominal** (sem ordem declarada) — e o pandas se recusa a comparar com `>`, lançando `TypeError`. É o tipo protegendo contra uma pergunta sem sentido: não existe categoria "maior" que outra quando não há ordem.

In [ ]:
nominal = pd.Series(["grande", "pequeno", "medio"], dtype="category")  # sem ordered=True

try:
    nominal.gt("pequeno")
except TypeError as erro:
    print("TypeError:", erro)

## 1.2 — Dados Retangulares

Apenas importa o pandas.

In [ ]:
import pandas as pd

Carrega os dados dos estados e mostra o formato do DataFrame — `(27, 4)`, ou seja, 27 registros (linhas) e 4 variáveis (colunas) — junto com as primeiras linhas da tabela.

In [ ]:
estado = pd.read_csv("https://raw.githubusercontent.com/BragaD/UnDF-Bases3-Estatistica-202602/main/dados/estados.csv")
print("linhas x colunas:", estado.shape)
estado.head()

`estado.info()` complementa o `.head()`: mostra quantos valores não nulos cada coluna tem e quanto de memória a tabela ocupa.

In [ ]:
estado.info()

Compara o índice numérico padrão (0, 1, 2...) com um índice significativo: trocar o índice pela sigla do estado transforma `.loc["SP"]` numa busca por rótulo, em vez de por posição.

In [ ]:
print("índice padrão:", estado.index[:5].tolist())

# Um índice significativo torna a busca por rótulo natural
por_estado = estado.set_index("Sigla")
por_estado.loc["SP"]

## 1.3 — Estimativas de Localização

Importa as bibliotecas usadas nesta seção — pandas, numpy, matplotlib, `trim_mean` para a média aparada e `wquantiles` para estimativas ponderadas — e fixa o tamanho padrão das figuras.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import trim_mean
import wquantiles
from formato import num

plt.rcParams["figure.figsize"] = (7, 4)

Calcula a média da população dos estados. Como a média soma todos os valores, o outlier populacional (São Paulo) puxa esse número para cima sozinho.

In [ ]:
media = estado["Populacao"].mean()
print(f"Média: {num(media)}")

Calcula a média aparada de 10%: descarta os 2 estados mais populosos e os 2 menos populosos antes de tirar a média, o que a torna resistente a esse tipo de outlier.

In [ ]:
media_aparada = trim_mean(estado["Populacao"], 0.1)
print(f"Média aparada (10%): {num(media_aparada)}")

Calcula a mediana da população e identifica a que estado ela corresponde. Com n = 27 (ímpar), a mediana é a população de um estado real — a Paraíba —, não uma conta entre dois valores.

In [ ]:
mediana = estado["Populacao"].median()
print(f"Mediana: {num(mediana)}")

# Qual estado é a mediana? Com n ímpar, ela é uma observação de verdade.
estado_mediano = estado.loc[estado["Populacao"] == mediana, "Estado"].iloc[0]
print(f"É a população de: {estado_mediano}")

Conta quantos valores distintos de população existem. Como as 27 populações são todas diferentes, nenhuma se repete — e por isso a moda simplesmente não tem o que dizer sobre essa variável.

In [ ]:
print(f"Valores únicos de população: {estado['Populacao'].nunique()} de {len(estado)}")

Carrega as causas de atraso de voo de Dallas/Fort Worth e identifica a categoria mais frequente. A moda é `VooAnterior` — o atraso mais comum não vem do clima nem da companhia, mas de um voo anterior da mesma aeronave chegando atrasado.

In [ ]:
dfw = pd.read_csv("https://raw.githubusercontent.com/BragaD/UnDF-Bases3-Estatistica-202602/main/dados/dfw_airline.csv").rename(columns={
    "Carrier": "Companhia", "ATC": "ControleAereo", "Weather": "Clima",
    "Security": "Seguranca", "Inbound": "VooAnterior"})
print(f"Causa modal de atraso: {dfw.iloc[0].idxmax()}")

Calcula a média simples, a média ponderada e a mediana ponderada da taxa de homicídios, usando a população como peso. A ponderada é **menor** que a simples porque São Paulo — o estado mais populoso e com a menor taxa — puxa o resultado para baixo quando pesado pela população.

In [ ]:
media_pond = np.average(estado["Taxa.Homicidios"], weights=estado["Populacao"])
mediana_pond = wquantiles.median(estado["Taxa.Homicidios"], weights=estado["Populacao"])

print(f"Média simples     : {num(estado['Taxa.Homicidios'].mean(), 4)}")
print(f"Média ponderada   : {num(media_pond, 4)}")
print(f"Mediana ponderada : {num(mediana_pond, 4)}")

Desenha o histograma da população com as três estimativas de localização sobrepostas. A média fica deslocada para a direita das outras duas; média aparada e mediana praticamente coincidem — é o que "robusto a extremos" significa na prática.

In [ ]:
fig, ax = plt.subplots()

ax.hist(estado["Populacao"] / 1e6, bins=20, color="#b0c4d8", edgecolor="white")
ax.axvline(media / 1e6, color="#c0392b", linestyle="-", linewidth=2, label=f"Média: {num(media/1e6, 1)}M")
ax.axvline(media_aparada / 1e6, color="#e67e22", linestyle="--", linewidth=2, label=f"Média aparada: {num(media_aparada/1e6, 1)}M")
ax.axvline(mediana / 1e6, color="#27ae60", linestyle=":", linewidth=2.5, label=f"Mediana: {num(mediana/1e6, 1)}M")

ax.set_xlabel("População (milhões)")
ax.set_ylabel("Número de estados")
ax.legend()
plt.tight_layout()
plt.show()

### Agora é com você — aluguéis em 5 cidades brasileiras

Até aqui os dados vieram prontos. Agora você recebe um arquivo e precisa **descobrir** o que tem dentro antes de calcular qualquer coisa: **10.692 imóveis para alugar** em São Paulo, Rio de Janeiro, Belo Horizonte, Porto Alegre e Campinas (dataset *Brazilian houses to rent*, domínio público — CC0). Só os nomes das colunas foram traduzidos; o que estava estranho no original continua estranho, de propósito.

Rode a célula abaixo para carregar o conjunto e faça as quatro tarefas **na ordem** — cada uma depende da anterior:

1. **Tamanho.** Quantas linhas e quantas colunas? Quais são os nomes das colunas? (`.shape`, `.columns`)
2. **Classificação.** Classifique **cada** coluna na taxonomia da seção 1.1 — numérico contínuo, numérico discreto, categórico nominal, categórico ordinal ou categórico binário — e justifique em uma frase. Há alguma ordinal?
3. **Inferência × classificação.** Veja o tipo que o pandas inferiu para cada coluna (`.dtypes`) e compare com a sua classificação. Alguma coluna foi lida com um tipo que **não** é o mais apropriado? Descubra **por quê** olhando os valores dela, e corrija.
4. **Localização.** Para **cada** coluna, calcule as medidas de localização que fazem sentido para o tipo dela — e só elas. Em quais colunas média e mediana ficaram muito diferentes, e o que isso indica?

As respostas comentadas estão no [site do livro, ao final da seção 1.3](https://BragaD.github.io/UnDF-Bases3-Estatistica-202602/content/cap01/03-estimativas-localizacao.html) — **tente antes de abrir**.

In [ ]:
# Agora é com você — carregue o conjunto e explore. As tarefas estão na célula acima.
import pandas as pd

alugueis = pd.read_csv("https://raw.githubusercontent.com/BragaD/UnDF-Bases3-Estatistica-202602/main/dados/alugueis.csv")
alugueis.head()

## 1.4 — Estimativas de Variabilidade

Importa as bibliotecas desta seção — incluindo `robust` da `statsmodels`, usada para o MAD — e recarrega os dados dos estados.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from statsmodels import robust
from formato import num

plt.rcParams["figure.figsize"] = (7, 4)
estado = pd.read_csv("https://raw.githubusercontent.com/BragaD/UnDF-Bases3-Estatistica-202602/main/dados/estados.csv")

Calcula o desvio-padrão da população dos estados, a medida de dispersão mais citada, mas também a mais sensível a valores extremos.

In [ ]:
desvio = estado["Populacao"].std()
print(f"Desvio-padrão: {num(desvio)}")

Calcula o IQR (amplitude interquartil): a diferença entre o 75º e o 25º percentil, uma medida de dispersão robusta que ignora o que acontece nas pontas.

In [ ]:
iqr = estado["Populacao"].quantile(0.75) - estado["Populacao"].quantile(0.25)
print(f"IQR: {num(iqr)}")

Calcula o MAD (desvio absoluto mediano) de duas formas — pela função pronta e manualmente — para mostrar de onde vem o fator 0,6745 que o calibra para ser comparável ao desvio-padrão sob uma distribuição normal.

In [ ]:
mad = robust.scale.mad(estado["Populacao"])
print(f"MAD: {num(mad)}")

# O mesmo cálculo, explicitamente — para mostrar de onde vem o 0,6745
mad_manual = abs(estado["Populacao"] - estado["Populacao"].median()).median() / 0.6744897501960817
print(f"MAD (manual): {num(mad_manual)}")

Desenha o histograma da população com desvio-padrão, IQR e MAD marcados a partir da mediana. O desvio-padrão é quase o dobro do MAD — o mesmo sinal de cauda longa à direita que a diferença entre média e mediana já indicava.

In [ ]:
mediana = estado["Populacao"].median()
fig, ax = plt.subplots()

ax.hist(estado["Populacao"] / 1e6, bins=20, color="#b0c4d8", edgecolor="white")
ax.axvline(mediana / 1e6, color="#2c3e50", linewidth=2, label=f"Mediana: {num(mediana/1e6, 1)}M")

for medida, valor, cor, estilo in [
    ("Desvio-padrão", desvio, "#c0392b", "-"),
    ("IQR", iqr, "#e67e22", "--"),
    ("MAD", mad, "#27ae60", ":"),
]:
    ax.axvline((mediana + valor) / 1e6, color=cor, linestyle=estilo, linewidth=2,
               label=f"{medida}: {num(valor/1e6, 1)}M")

ax.set_xlabel("População (milhões)")
ax.set_ylabel("Número de estados")
ax.legend()
plt.tight_layout()
plt.show()

Calcula os z-scores da população e da taxa de homicídios de São Paulo. Os dois têm sinais opostos — SP é um extremo em população (bem acima da média) e um extremo oposto em taxa de homicídios (bem abaixo) —, e por serem adimensionais, podem ser comparados diretamente mesmo vindo de escalas completamente diferentes.

In [ ]:
mu, sd = estado["Populacao"].mean(), estado["Populacao"].std(ddof=1)
sp = estado.loc[estado["Sigla"] == "SP", "Populacao"].iloc[0]
z_sp = (sp - mu) / sd
print(f"z-score da população de SP: {num(z_sp, 2)}")

mu_t, sd_t = estado["Taxa.Homicidios"].mean(), estado["Taxa.Homicidios"].std(ddof=1)
sp_t = estado.loc[estado["Sigla"] == "SP", "Taxa.Homicidios"].iloc[0]
print(f"z-score da taxa de homicídios de SP: {num((sp_t - mu_t) / sd_t, 2)}")

### Agora é com você — quanto os aluguéis se espalham

Os mesmos 10.692 imóveis da seção 1.3 — agora a pergunta é **quanto os valores se espalham** em torno do centro. Concentre-se nas seis colunas contínuas (`area_m2`, `condominio`, `aluguel`, `iptu`, `seguro_incendio`, `total`). Rode a célula abaixo e faça as tarefas **na ordem**:

1. **Três medidas.** Para cada coluna, calcule desvio-padrão, IQR e MAD (`robust.scale.mad` já vem calibrado pelo 0,6745) e monte uma tabela com as três lado a lado.
2. **Detector de outlier.** Divida o desvio-padrão pelo MAD em cada coluna. Numa normal, os dois são parecidos. Em quais colunas a razão explode — e o que isso indica?
3. **Robustez na prática.** Remova o(s) imóvel(is) com o maior `condominio` e recalcule as três medidas dessa coluna. Qual mudou mais? Qual quase não se mexeu?
4. **Escores-padrão.** Calcule o z do maior aluguel e conte quantos imóveis têm |z| > 3. Se fosse normal, que fração você esperaria? O que a diferença diz sobre a régua?

As respostas comentadas estão no [site do livro, ao final da seção 1.4](https://BragaD.github.io/UnDF-Bases3-Estatistica-202602/content/cap01/04-estimativas-variabilidade.html) — **tente antes de abrir**.

In [ ]:
# Agora é com você — os mesmos aluguéis da 1.3, agora medindo dispersão. As tarefas estão na célula acima.
import pandas as pd
from statsmodels import robust

alugueis = pd.read_csv("https://raw.githubusercontent.com/BragaD/UnDF-Bases3-Estatistica-202602/main/dados/alugueis.csv")
continuas = ["area_m2", "condominio", "aluguel", "iptu", "seguro_incendio", "total"]
alugueis[continuas].head()

## 1.5 — Explorando a Distribuição dos Dados

Importa as bibliotecas desta seção, incluindo `scipy.stats`, e recarrega os dados dos estados.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats
from formato import num

plt.rcParams["figure.figsize"] = (7, 4)
estado = pd.read_csv("https://raw.githubusercontent.com/BragaD/UnDF-Bases3-Estatistica-202602/main/dados/estados.csv")

Calcula os percentis 5, 25, 50, 75 e 95 da taxa de homicídios — a generalização da mediana para qualquer fração dos dados, não só a metade.

In [ ]:
percentis = [0.05, 0.25, 0.5, 0.75, 0.95]
tabela = pd.DataFrame(estado["Taxa.Homicidios"].quantile(percentis))
tabela.index = [f"{p:.0%}" for p in percentis]
tabela.transpose().map(lambda v: num(v, 3))

Desenha o boxplot da população: a caixa vai do primeiro ao terceiro quartil (a mesma largura do IQR), a linha central é a mediana, e os pontos isolados acima do bigode são os estados atipicamente populosos (Minas Gerais e São Paulo).

In [ ]:
fig, ax = plt.subplots(figsize=(4, 5))
(estado["Populacao"] / 1e6).plot.box(ax=ax)
ax.set_ylabel("População (milhões)")
plt.tight_layout()
plt.show()

Desenha o gráfico de violino da população — a mesma caixa central, mas com a densidade estimada desenhada dos dois lados, revelando a forma inteira da distribuição, não só cinco números.

In [ ]:
fig, ax = plt.subplots(figsize=(4, 5))
ax.violinplot(estado["Populacao"] / 1e6, showmedians=True)
ax.set_ylabel("População (milhões)")
ax.set_xticks([])
plt.tight_layout()
plt.show()

Divide a população em 10 faixas de largura igual e conta quantos estados caem em cada uma. Mais da metade dos estados se concentra na primeira faixa — o retrato de uma distribuição com cauda longa à direita, não um defeito do método.

In [ ]:
faixas = pd.cut(estado["Populacao"], 10)
faixas.value_counts().sort_index()

Desenha o histograma da população em 10 faixas — a mesma tabela de frequência anterior, agora como gráfico, com barras encostadas porque o eixo é contínuo.

In [ ]:
fig, ax = plt.subplots()
(estado["Populacao"] / 1e6).plot.hist(ax=ax, bins=10, edgecolor="white")
ax.set_xlabel("População (milhões)")
ax.set_ylabel("Número de estados")
plt.tight_layout()
plt.show()

Desenha o histograma da taxa de homicídios (normalizado por `density=True`) com a curva de densidade sobreposta — as duas na mesma escala, com área total igual a 1.

In [ ]:
fig, ax = plt.subplots()
estado["Taxa.Homicidios"].plot.hist(ax=ax, density=True, xlim=[0, 40],
                                bins=range(0, 40, 2), edgecolor="white")
estado["Taxa.Homicidios"].plot.density(ax=ax, linewidth=2)
ax.set_xlabel("Taxa de homicídios (por 100.000)")
plt.tight_layout()
plt.show()

Constrói três turmas fictícias de 25 alunos com **exatamente** a mesma média (65) e a mesma variância, mas formas diferentes: uma com cauda à direita, seu espelho com cauda à esquerda, e uma simétrica por construção — para mostrar que média e variância nada dizem sobre a forma da distribuição.

In [ ]:
N, MEDIA, DESVIO = 25, 65.0, 12.0

def padroniza(x, media=MEDIA, desvio=DESVIO):
    """Crava média e desvio exatos, via z-score e reescala.

    A assimetria é invariante a transformação linear — ela sobrevive intacta.
    É isso que permite três conjuntos com a MESMA média e a MESMA variância,
    mas formas diferentes.
    """
    x = np.asarray(x, float)
    return (x - x.mean()) / x.std(ddof=1) * desvio + media

# Um gerador por turma: a ordem de consumo do RNG não pode alterar um conjunto
# em silêncio quando outro mudar.
turma_a = padroniza(np.random.default_rng(42).lognormal(0, 1.0, N))

# Espelho em torno da média: preserva média e variância, e inverte o SINAL da
# assimetria — exatamente, não aproximadamente.
turma_b = 2 * MEDIA - turma_a

# Simétrica POR CONSTRUÇÃO, não por amostragem: 12 desvios, seus 12 espelhos,
# e o centro. A assimetria é exatamente zero, não "próxima de zero".
desvios = np.abs(np.random.default_rng(7).normal(0, 1, (N - 1) // 2))
turma_c = padroniza(np.concatenate([-desvios, [0.0], desvios]))

turmas = {"A — à direita": turma_a, "B — à esquerda": turma_b, "C — simétrica": turma_c}

# As garantias são verificadas, não afirmadas.
assert np.allclose([t.mean() for t in turmas.values()], MEDIA)
assert np.allclose([t.var(ddof=1) for t in turmas.values()], DESVIO ** 2)
assert np.isclose(stats.skew(turma_a, bias=False), -stats.skew(turma_b, bias=False))
assert np.isclose(stats.skew(turma_c, bias=False), 0, atol=1e-12)
assert all(t.min() >= 0 and t.max() <= 100 for t in turmas.values())

Monta uma tabela com média, mediana, variância e assimetria das três turmas. Assimetria positiva vem com média maior que a mediana; negativa, com média menor; zero, com as duas iguais — a regra aparece sozinha nos números.

In [ ]:
resumo = pd.DataFrame(
    {
        "Média": [num(t.mean(), 1) for t in turmas.values()],
        "Mediana": [num(np.median(t), 1) for t in turmas.values()],
        "Variância": [num(t.var(ddof=1), 1) for t in turmas.values()],
        "Assimetria": [num(stats.skew(t, bias=False), 2) for t in turmas.values()],
    },
    index=list(turmas),
)
resumo

Desenha o histograma das três turmas lado a lado, com média (vermelho) e mediana (verde) marcadas — o rastro de notas na cauda é o que separa as duas linhas nas turmas assimétricas.

In [ ]:
fig, eixos = plt.subplots(1, 3, figsize=(11, 3.3), sharey=True)

for ax, (nome, notas) in zip(eixos, turmas.items()):
    ax.hist(notas, bins=np.arange(30, 101, 7), color="#b0c4d8", edgecolor="white")
    ax.axvline(notas.mean(), color="#c0392b", linewidth=2)
    ax.axvline(np.median(notas), color="#27ae60", linestyle="--", linewidth=2)
    ax.set_title(nome, fontsize=10)
    ax.set_xlabel("Nota")

eixos[0].set_ylabel("Alunos")
plt.tight_layout()
plt.show()

Desenha as mesmas três turmas em boxplot horizontal. Nas turmas assimétricas, a mediana fica deslocada para um lado da caixa e um bigode é bem mais longo que o outro; na simétrica, tudo fica centrado.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 3))

caixas = ax.boxplot(
    [turma_c, turma_b, turma_a],          # de baixo para cima: simétrica, esquerda, direita
    tick_labels=["C — simétrica", "B — à esquerda", "A — à direita"],
    vert=False,
    patch_artist=True,
    medianprops={"color": "#27ae60", "linewidth": 2},
    widths=0.6,
)
for caixa in caixas["boxes"]:
    caixa.set(facecolor="#b0c4d8", edgecolor="#2c3e50")

ax.set_xlabel("Nota")
plt.tight_layout()
plt.show()

Calcula a assimetria da população e da taxa de homicídios dos estados reais. A população tem assimetria forte e positiva (cauda à direita, por causa de São Paulo); a taxa de homicídios está quase simétrica.

In [ ]:
print(f"População das UFs  : {num(stats.skew(estado['Populacao'], bias=False), 2)}")
print(f"Taxa de homicídios : {num(stats.skew(estado['Taxa.Homicidios'], bias=False), 2)}")

Calcula a curtose (excesso, relativo à normal) da população e da taxa de homicídios. A população tem curtose muito alta — de novo, o efeito de São Paulo como extremo isolado —; a taxa de homicídios tem curtose negativa, caudas mais leves que a normal.

In [ ]:
print(f"Curtose da população das UFs : {num(stats.kurtosis(estado['Populacao']), 2)}")
print(f"Curtose da taxa de homicídios: {num(stats.kurtosis(estado['Taxa.Homicidios']), 2)}")

### Agora é com você — a forma da distribuição dos aluguéis

De volta aos 10.692 aluguéis das seções 1.3 e 1.4. Você já sabe onde está o centro e quanto os valores se espalham; falta a **forma**. Rode a célula abaixo e faça as tarefas **na ordem**:

1. **Percentis.** Calcule os percentis 5, 25, 50, 75 e 95 de `aluguel` e de `condominio`, com o máximo de cada uma na última linha. O que a distância entre o P95 e o máximo diz?
2. **Boxplot.** Desenhe o boxplot de `aluguel`. Pela regra de 1,5×IQR, calcule o limite superior do bigode e conte os outliers. Que fração isso representa? Há outliers pelo lado de baixo?
3. **Histograma que não funciona.** Monte a tabela de frequência de `area_m2` em 10 faixas (`pd.cut`) e o histograma. Por que ficaram inúteis? Refaça só com os imóveis até o percentil 99 da área e compare.
4. **Assimetria.** Calcule a assimetria (`stats.skew`) de `aluguel`, `condominio` e `area_m2`, com média e mediana ao lado. O sinal bate com "média > mediana ⇒ assimetria positiva"? Qual é a mais assimétrica, e por quê?

As respostas comentadas estão no [site do livro, ao final da seção 1.5](https://BragaD.github.io/UnDF-Bases3-Estatistica-202602/content/cap01/05-distribuicao-dados.html) — **tente antes de abrir**.

In [ ]:
# Agora é com você — os mesmos aluguéis, agora olhando a forma da distribuição. As tarefas estão na célula acima.
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats

alugueis = pd.read_csv("https://raw.githubusercontent.com/BragaD/UnDF-Bases3-Estatistica-202602/main/dados/alugueis.csv")
alugueis[["aluguel", "condominio", "area_m2"]].head()

## 1.6 — Explorando Dados Binários e Categóricos

Importa as bibliotecas desta seção e fixa o tamanho padrão das figuras.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from formato import num

plt.rcParams["figure.figsize"] = (7, 4)

Carrega as causas de atraso de voo e calcula a proporção de cada uma sobre o total — o resumo que substitui a média quando o dado é categórico, já que não há como somar ou ordenar categorias.

In [ ]:
dfw = pd.read_csv("https://raw.githubusercontent.com/BragaD/UnDF-Bases3-Estatistica-202602/main/dados/dfw_airline.csv").rename(columns={
    "Carrier": "Companhia",
    "ATC": "ControleAereo",
    "Weather": "Clima",
    "Security": "Seguranca",
    "Inbound": "VooAnterior",
})
proporcoes = 100 * dfw / dfw.values.sum()
proporcoes.round(2).map(lambda v: num(v, 2))

Desenha um gráfico de barras das causas de atraso. As barras vêm separadas de propósito: o eixo é categórico, não contínuo, e não existe uma ordem natural entre as causas.

In [ ]:
fig, ax = plt.subplots()
dfw.transpose().plot.bar(ax=ax, legend=False, color="#4a90a4", edgecolor="white")
ax.set_xlabel("Causa do atraso")
ax.set_ylabel("Número de atrasos")
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

Desenha a taxa de homicídios dos nove estados do Nordeste com o eixo Y começando em 20, não em zero — um truque comum que faz as diferenças parecerem muito maiores do que realmente são.

In [ ]:
estado = pd.read_csv("https://raw.githubusercontent.com/BragaD/UnDF-Bases3-Estatistica-202602/main/dados/estados.csv")
nordeste = ["AL", "BA", "CE", "MA", "PB", "PE", "PI", "RN", "SE"]
ne = estado[estado["Sigla"].isin(nordeste)].sort_values("Sigla")

fig, ax = plt.subplots(figsize=(7, 4))
ax.bar(ne["Sigla"], ne["Taxa.Homicidios"], color="#b0c4d8", edgecolor="white")
ax.set_ylim(20, 38)
ax.set_xlabel("Estado")
ax.set_ylabel("Taxa de homicídios (por 100 mil)")
plt.tight_layout()
plt.show()

Repete o mesmo gráfico com o eixo Y começando em zero — a versão honesta. A diferença entre o estado mais e o menos violento cai de uma impressão de ~36 vezes para a razão verdadeira, cerca de 1,8 vez.

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
ax.bar(ne["Sigla"], ne["Taxa.Homicidios"], color="#b0c4d8", edgecolor="white")
ax.set_ylim(0, 38)
ax.set_xlabel("Estado")
ax.set_ylabel("Taxa de homicídios (por 100 mil)")
plt.tight_layout()
plt.show()

Identifica a categoria mais frequente (a moda) entre as causas de atraso — a única medida de localização que sobra quando não há ordem nem distância entre as categorias.

In [ ]:
moda = proporcoes.transpose().iloc[:, 0].idxmax()
print(f"Moda (causa mais frequente): {moda}")

Calcula o valor esperado do custo por atraso, ponderando o custo de cada causa pela sua probabilidade. O resultado combina frequência e consequência num único número — a base de decisão sob incerteza.

In [ ]:
# Uma companhia estima o custo médio de compensação por passageiro,
# conforme a causa do atraso.
custo = {"Companhia": 180, "ControleAereo": 40, "Clima": 0, "Seguranca": 25, "VooAnterior": 120}

p = (dfw / dfw.values.sum()).transpose().iloc[:, 0]   # probabilidade de cada causa
ve = sum(p[causa] * valor for causa, valor in custo.items())

print(f"Valor esperado do custo por atraso: R$ {num(ve, 2)}")

## 1.7 — Correlação

Importa as bibliotecas desta seção, incluindo o seaborn para os gráficos de correlação.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from formato import num

plt.rcParams["figure.figsize"] = (7, 4)

Carrega preços de ações do S&P 500 e filtra as empresas de telecomunicações a partir de julho de 2012, montando uma tabela de 754 dias por 5 empresas para comparar seus retornos diários.

In [ ]:
SETORES = {
    "consumer_discretionary": "consumo_discricionario",
    "consumer_staples": "consumo_essencial",
    "energy": "energia",
    "etf": "etf",
    "financials": "financeiro",
    "health_care": "saude",
    "industrials": "industrial",
    "information_technology": "tecnologia_da_informacao",
    "materials": "materiais",
    "telecommunications_services": "telecomunicacoes",
    "utilities": "utilidade_publica",
}
setores = pd.read_csv("https://raw.githubusercontent.com/BragaD/UnDF-Bases3-Estatistica-202602/main/dados/sp500_sectors.csv").rename(
    columns={"sector": "Setor", "symbol": "Simbolo"}
)
setores["Setor"] = setores["Setor"].replace(SETORES)
precos  = pd.read_csv("https://raw.githubusercontent.com/BragaD/UnDF-Bases3-Estatistica-202602/main/dados/sp500_data.csv.gz", index_col=0)

simbolos_telecom = setores[setores["Setor"] == "telecomunicacoes"]["Simbolo"]
telecom = precos.loc[precos.index >= "2012-07-01", simbolos_telecom]

print("dias x empresas:", telecom.shape)
telecom.head()

Calcula a matriz de correlação de Pearson entre os retornos diários das cinco empresas de telecomunicações — simétrica, com 1 na diagonal.

In [ ]:
telecom.corr().round(3).map(lambda v: num(v, 3))

Desenha o gráfico de dispersão dos retornos diários de AT&T contra Verizon. A nuvem alongada na diagonal é o sinal visual da correlação positiva entre as duas (r ≈ 0,68).

In [ ]:
fig, ax = plt.subplots(figsize=(5, 5))
ax.scatter(telecom["T"], telecom["VZ"], alpha=0.5, s=20, color="#2c7fb8")
ax.axhline(0, color="grey", linewidth=0.8)
ax.axvline(0, color="grey", linewidth=0.8)
ax.set_xlabel("Retorno diário — AT&T (T)")
ax.set_ylabel("Retorno diário — Verizon (VZ)")
plt.tight_layout()
plt.show()

Desenha um heatmap da correlação entre 17 ETFs, com paleta divergente centrada em zero — a única forma prática de enxergar o padrão numa matriz grande demais para ler em números.

In [ ]:
etfs = precos.loc[precos.index > "2012-07-01",
                  setores[setores["Setor"] == "etf"]["Simbolo"]]

fig, ax = plt.subplots(figsize=(7, 6))
sns.heatmap(etfs.corr(), vmin=-1, vmax=1,
            cmap=sns.diverging_palette(20, 220, as_cmap=True), ax=ax)
plt.tight_layout()
plt.show()

Calcula a correlação entre $x$ e $x^2$: apesar de $y$ ser completamente determinado por $x$, a correlação de Pearson sai praticamente zero, porque ela só mede associação **linear** — a parábola simétrica cancela qualquer tendência linear.

In [ ]:
x = np.linspace(-1, 1, 200)
y = x ** 2

print(f"Correlação entre x e x²: {np.corrcoef(x, y)[0, 1]:.3e}")

## 1.8 — Explorando Duas ou Mais Variáveis

Importa as bibliotecas desta seção, incluindo o seaborn.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from formato import num

plt.rcParams["figure.figsize"] = (7, 4)

Carrega avaliações fiscais de imóveis do condado de King e filtra valores e áreas extremas — não uma correção estatística, mas uma decisão de visualização, para que o grosso dos dados não fique comprimido num canto do gráfico.

In [ ]:
kc = pd.read_csv("https://raw.githubusercontent.com/BragaD/UnDF-Bases3-Estatistica-202602/main/dados/kc_tax.csv.gz").rename(columns={
    "TaxAssessedValue": "ValorVenal",
    "SqFtTotLiving": "AreaConstruida",
    "ZipCode": "CEP",
})
print(f"registros brutos: {num(len(kc), 0)}")

kc0 = kc.loc[(kc.ValorVenal < 750000) &
             (kc.AreaConstruida > 100) &
             (kc.AreaConstruida < 3500), :]
print(f"após filtrar extremos: {num(len(kc0), 0)}")

Desenha um hexbin de área construída contra valor venal: em vez de um ponto por imóvel (impraticável com 432 mil registros), cada hexágono é colorido pela quantidade de imóveis que caem nele.

In [ ]:
fig, ax = plt.subplots(figsize=(6, 5))
kc0.plot.hexbin(x="AreaConstruida", y="ValorVenal",
                gridsize=30, sharex=False, ax=ax)
ax.set_xlabel("Área construída (pés²)")
ax.set_ylabel("Valor venal (US$)")
plt.tight_layout()
plt.show()

Desenha as curvas de nível (KDE) da mesma relação, sobre uma amostra de 10.000 imóveis com semente fixa — uma versão mais suave do hexbin, sem custar minutos de processamento.

In [ ]:
amostra = kc0.sample(10000, random_state=42)

fig, ax = plt.subplots(figsize=(6, 5))
sns.kdeplot(data=amostra, x="AreaConstruida", y="ValorVenal", ax=ax)
ax.set_xlabel("Área construída (pés²)")
ax.set_ylabel("Valor venal (US$)")
plt.tight_layout()
plt.show()

Monta a tabela de contingência cruzando nota de risco do empréstimo e situação atual, em contagens brutas. Comparar essas contagens diretamente engana, porque grupos de tamanhos muito diferentes (como as notas B e C) sempre têm mais de tudo.

In [ ]:
SITUACAO = {
    "Fully Paid": "Quitado",
    "Current": "Em dia",
    "Late": "Atrasado",
    "Charged Off": "Inadimplente",
}
lc = pd.read_csv("https://raw.githubusercontent.com/BragaD/UnDF-Bases3-Estatistica-202602/main/dados/lc_loans.csv").rename(columns={"status": "Situacao", "grade": "Nota"})
lc["Situacao"] = lc["Situacao"].replace(SITUACAO)

contagem = lc.pivot_table(index="Nota", columns="Situacao",
                          aggfunc=lambda x: len(x), margins=True)
contagem

Normaliza a tabela de contingência por linha, convertendo contagens em proporções. Feito isso, a taxa de inadimplência cresce de forma monótona da nota A até a G — o padrão que a contagem bruta escondia.

In [ ]:
prop = contagem.copy().loc["A":"G", :].astype(float)
situacoes = [c for c in prop.columns if c != "All"]
prop.loc[:, situacoes] = prop.loc[:, situacoes].div(prop["All"], axis=0)
prop["All"] = prop["All"] / sum(prop["All"])
prop.round(3).map(lambda v: num(v, 3))

Desenha um boxplot do percentual diário de atrasos atribuíveis à companhia, um por companhia aérea. As medianas ordenam claramente as companhias, da Alaska (melhor) à American (pior).

In [ ]:
voos = pd.read_csv("https://raw.githubusercontent.com/BragaD/UnDF-Bases3-Estatistica-202602/main/dados/airline_stats.csv").rename(columns={
    "pct_carrier_delay": "pct_atraso_companhia",
    "pct_atc_delay": "pct_atraso_controle",
    "pct_weather_delay": "pct_atraso_clima",
    "airline": "Companhia",
})

fig, ax = plt.subplots(figsize=(7, 5))
voos.boxplot(by="Companhia", column="pct_atraso_companhia", ax=ax)
ax.set_xlabel("")
ax.set_ylabel("% diário de voos atrasados")
ax.set_ylim(0, 50)
plt.suptitle("")
plt.title("")
plt.xticks(rotation=20)
plt.tight_layout()
plt.show()

Desenha o mesmo dado em violin plot. A forma revela o que o boxplot esconde — como a concentração de dias com atraso quase zero na Alaska, que aparece como um bojo perto do eixo.

In [ ]:
fig, ax = plt.subplots(figsize=(7, 5))
sns.violinplot(data=voos, x="Companhia", y="pct_atraso_companhia",
               ax=ax, inner="quartile", color="#b0c4d8")
ax.set_xlabel("")
ax.set_ylabel("% diário de voos atrasados")
ax.set_ylim(0, 50)
plt.xticks(rotation=20)
plt.tight_layout()
plt.show()

Desenha um `FacetGrid` de hexbins, um painel por CEP, condicionando a relação entre área e valor a uma terceira variável — o bairro. A inclinação da relação muda de painel para painel, algo que uma única superfície agregada não revelaria.

In [ ]:
kc_ceps = kc0.loc[kc0.CEP.isin([98188, 98105, 98108, 98126]), :]

def hexbin(x, y, color, **kwargs):
    cmap = sns.light_palette(color, as_cmap=True)
    plt.hexbin(x, y, gridsize=25, cmap=cmap, **kwargs)

g = sns.FacetGrid(kc_ceps, col="CEP", col_wrap=2, height=3.2)
g.map(hexbin, "AreaConstruida", "ValorVenal", extent=[0, 3500, 0, 700000])
g.set_axis_labels("Área construída (pés²)", "Valor venal (US$)")
plt.tight_layout()
plt.show()